In [ ]:
import os
import random
import shutil
import pickle
from astropy.io import fits

# Directorios de origen y destino
src_folder = "spectrums"
dst_folder = "spectrums training 100k"

# Crear el directorio de destino si no existe
if not os.path.exists(dst_folder):
    os.makedirs(dst_folder)

# Definir los dos intervalos de redshift (ejemplo)
target_range1 = (0.1, 2)   # Primer intervalo: de 0.1 a 1
target_range2 = (0, 0.1)   # Segundo intervalo: de 2.1 a 3

# Número de archivos a copiar (total de ambos intervalos)
num_files_to_add = 20000

# Obtener la lista de todos los archivos .fits en el directorio de origen
all_files = [file for file in os.listdir(src_folder) if file.lower().endswith(".fits")]

# Randomizar la lista de archivos
random.shuffle(all_files)

# Diccionario para almacenar los redshifts de los archivos que se seleccionen
redshift_dict = {}

# Lista para almacenar los archivos seleccionados
selected_files = []
count = 0

# Recorrer la lista randomizada hasta alcanzar el número deseado
for file in all_files:
    src_path = os.path.join(src_folder, file)
    try:
        with fits.open(src_path) as hdul:
            redshift = hdul[2].data["Z"][0]
            # Verificar si el redshift se encuentra en alguno de los intervalos definidos
            if (target_range1[0] <= redshift < target_range1[1]) or (target_range2[0] <= redshift < target_range2[1]):
                dst_path = os.path.join(dst_folder, file)
                # Si el archivo ya existe en el destino, se "borra" el redshift (es decir, no se agrega)
                if os.path.exists(dst_path):
                    if file in redshift_dict:
                        del redshift_dict[file]
                    continue  # Omitir el archivo ya existente
                # Agregar el archivo a la lista de selección y almacenar su redshift
                selected_files.append(file)
                redshift_dict[file] = redshift
                count += 1
                if count % 100 == 0:
                    print(f"{count} archivos seleccionados...")
                if count >= num_files_to_add:
                    break
    except Exception as e:
        print(f"Error procesando {file}: {e}")

# Verificar si se han encontrado suficientes archivos
if len(selected_files) < num_files_to_add:
    print(f"Se encontraron solo {len(selected_files)} archivos válidos en los intervalos especificados, se requieren {num_files_to_add}.")
    exit(1)
else:
    print(f"Se seleccionaron {len(selected_files)} archivos que cumplen con los intervalos requeridos.")

# Copiar los archivos seleccionados al directorio de destino
for file in selected_files:
    src_path = os.path.join(src_folder, file)
    dst_path = os.path.join(dst_folder, file)
    shutil.copy(src_path, dst_path)
    print(f"Copiado: {file}")

print("Proceso de copia completado.")

# Guardar/actualizar los nuevos redshifts en un archivo pickle
new_redshifts_file = "extra/new_redshifts.pkl"

with open(new_redshifts_file, "wb") as f:
    pickle.dump(redshift_dict, f)

print(f"Se han guardado los nuevos redshifts en '{new_redshifts_file}'.")

100 archivos seleccionados...
200 archivos seleccionados...
300 archivos seleccionados...
400 archivos seleccionados...
500 archivos seleccionados...
600 archivos seleccionados...
700 archivos seleccionados...
800 archivos seleccionados...
900 archivos seleccionados...
1000 archivos seleccionados...
1100 archivos seleccionados...
1200 archivos seleccionados...
1300 archivos seleccionados...
1400 archivos seleccionados...
1500 archivos seleccionados...
1600 archivos seleccionados...
1700 archivos seleccionados...
1800 archivos seleccionados...
1900 archivos seleccionados...
2000 archivos seleccionados...
2100 archivos seleccionados...
2200 archivos seleccionados...
2300 archivos seleccionados...
2400 archivos seleccionados...
2500 archivos seleccionados...
2600 archivos seleccionados...
2700 archivos seleccionados...
2800 archivos seleccionados...
2900 archivos seleccionados...
3000 archivos seleccionados...
3100 archivos seleccionados...
3200 archivos seleccionados...
3300 archivos sel

In [ ]:
import os
import pickle
import numpy as np
import matplotlib.pyplot as plt

# Cargar el array original de redshifts desde el pickle
with open("extra/redshift_array.pkl", "rb") as f:
    redshift_array = pickle.load(f)

# Cargar los nuevos redshifts desde otro archivo pickle
# Se asume que previamente has guardado los nuevos redshifts en "new_redshifts.pkl"
if os.path.exists("extra/new_redshifts.pkl"):
    with open("extra/new_redshifts.pkl", "rb") as f:
        new_redshifts = pickle.load(f)
else:
    print("El archivo 'new_redshifts.pkl' no existe. Asegúrate de haberlo generado.")
    new_redshifts = np.array([])

# Combinar el conjunto original con los nuevos redshifts
if new_redshifts.size > 0:
    redshift_array_updated = np.concatenate((redshift_array, new_redshifts))
else:
    redshift_array_updated = redshift_array

print(f"Total de redshifts actualizados: {redshift_array_updated.size}")

# Guardar el array actualizado (opcional)
with open("redshift_array_updated.pkl", "wb") as f:
    pickle.dump(redshift_array_updated, f)
print("Guardado el array actualizado en 'redshift_array_updated.pkl'.")

# Definir los bins con intervalos de 0.1 y dibujar el histograma actualizado
bins = np.arange(0, redshift_array_updated.max() + 0.1, 0.1)
plt.figure(figsize=(18,16))
plt.hist(redshift_array_updated, bins=bins, color='skyblue', edgecolor='black')
plt.xlabel("Redshift")
plt.ylabel("Número de archivos")
plt.title("Histograma de redshifts actualizado (bin width = 0.1)")
plt.xlim(0, 7.1)
plt.show()

# Guardar el array de redshifts en un archivo pickle
with open("extra/redshift_array.pkl", "wb") as f:
    pickle.dump(redshift_array_updated, f)

FileNotFoundError: [Errno 2] No such file or directory: 'redshift_array.pkl'

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits

# Directorio que contiene los archivos FITS
dst_folder = "spectrums training 100k"

plt.figure(figsize=(18,16))
# Dibujar todos los puntos: cada punto se dibuja en (redshift, redshift)
plt.scatter(redshift_array_updated, redshift_array_updated, color='blue', alpha=0.5, label='Puntos de redshift')

# Dibujar la función y = x
x_vals = np.linspace(0, redshift_array_updated.max(), 100)
plt.plot(x_vals, x_vals, color='red', linestyle='--', label='y = x')

plt.xlabel("Redshift")
plt.ylabel("Redshift")
plt.title("Representación de puntos de redshift y la función y = x")
plt.legend()
plt.show()